# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR⁲ dataset using the `mlcroissant` library. All dataset schema elements are referenced by their Croissant `@id` fields for consistency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
if hasattr(metadata, 'record_sets'):
    print("Available record sets:")
    for rs in metadata.record_sets:
        print(f"  RecordSet @id: {rs.id}  |  name: {rs.name if hasattr(rs, 'name') else '-'}")
        if hasattr(rs, 'fields'):
            for fld in rs.fields:
                print(f"    Field @id: {fld.id}  |  name: {fld.name if hasattr(fld, 'name') else '-'}  |  dataType: {fld.data_type if hasattr(fld, 'data_type') else '-'}")
else:
    # Fallback for cases when 'record_sets' is not directly available
    print("No record sets found in metadata.")

## 3. Data Extraction
Extract and load the records of each record set into pandas DataFrames for further analysis. All access is done using Croissant `@id` values.

In [ ]:
# Gather all record set @id values
record_sets_ids = []
if hasattr(metadata, 'record_sets'):
    record_sets_ids = [rs.id for rs in metadata.record_sets]
else:
    print("No record_sets found in metadata.")
# Display the list of record set IDs
print("RecordSet @ids:", record_sets_ids)

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# Pick the first record set for demonstration
active_record_set_id = record_sets_ids[0] if len(record_sets_ids) > 0 else None
if active_record_set_id is not None:
    print(f"Using record set: {active_record_set_id}")
    print(dataframes[active_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, including filtering, normalization, and grouping of numeric fields. All operations reference the dataset using Croissant `@id`s for columns.

In [ ]:
import numpy as np

# Pick a numeric field for analysis
# To follow the template, extract fields for the currently active record set
numeric_field_id = None
group_field_id = None

if hasattr(metadata, 'record_sets') and active_record_set_id:
    current_rs = next((rs for rs in metadata.record_sets if rs.id == active_record_set_id), None)
    # Try to find a field whose dataType is numeric
    if current_rs and hasattr(current_rs, 'fields'):
        for fld in current_rs.fields:
            if hasattr(fld, 'data_type') and (fld.data_type in ['schema:Float', 'schema:Number', 'schema:Integer', 'Float', 'Number', 'Integer']):
                numeric_field_id = fld.id
                break
        
        # Also select a text/categorical field (not the numeric field) for grouping if available
        for fld in current_rs.fields:
            if fld.id != numeric_field_id and hasattr(fld, 'data_type') and (fld.data_type in ['schema:Text', 'Text']):
                group_field_id = fld.id
                break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Proceed only if field exists and present in dataframe
if numeric_field_id and numeric_field_id in dataframes[active_record_set_id].columns:
    df = dataframes[active_record_set_id].copy()
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Drop NAs
    df = df.dropna(subset=[numeric_field_id])
    threshold = df[numeric_field_id].quantile(0.95)  # Filter top 5% as example
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields. All axis labels use the Croissant field `@id` for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in dataframes[active_record_set_id].columns:
    df = dataframes[active_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.xlabel(numeric_field_id + " (@id)")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xlabel(group_field_id + " (@id)")
        plt.ylabel(numeric_field_id + " (@id)")
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

The FAIR⁲ dataset was loaded, explored, and visualized via its Croissant schema using the `mlcroissant` library. By referencing all dataset components by their Croissant `@id` fields, this workflow ensures that all analyses are robust to schema changes and reproducible across future dataset versions.